# Meteosat-9 reflectance animation around Maroantsetra flood dates

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johrosa/srwi/blob/main/wekeo_meteosat_animation_maroantsetra_colab.ipynb)

This notebook builds Meteosat-9 IODC cloud-motion animations around Maroantsetra using **visible reflectance / natural-colour EUMETView WMS layers**, not rainfall products.

Default layer: `msg_iodc:vis006`, the Meteosat SEVIRI visible 0.6 micrometer channel. You can switch to `msg_iodc:rgb_natural` or `msg_iodc:rgb_eview` for rendered RGB imagery.

## 1. Install and imports

In [ ]:
!pip -q install requests pillow imageio hda

In [ ]:
import getpass
import json
import math
import os
import re
import time
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
from pathlib import Path

import imageio.v2 as imageio
import pandas as pd
import requests
from IPython.display import Image, display
from PIL import Image as PILImage, ImageDraw, ImageFont

## 2. Parameters

In [ ]:
MAROANTSETRA_LON = 49.7333
MAROANTSETRA_LAT = -15.4333
BBOX = [47.4, -17.6, 52.1, -13.2]  # west, south, east, north

EUMETVIEW_WMS_URL = "https://view.eumetsat.int/geoserver/wms"
REFLECTANCE_LAYER = "msg_iodc:vis006"
# Other useful non-rainfall layers:
# - msg_iodc:rgb_natural
# - msg_iodc:rgb_eview
# - msg_iodc:rgb_microphysics
# - msg_iodc:ir108 for day/night thermal cloud tracking, not reflectance

FLOOD_EVENTS = [
    {
        "name": "Cyclone Herold - Maroantsetra floods",
        "start": "2020-03-13T00:00:00Z",
        "end": "2020-03-18T23:59:59Z",
    },
    {
        "name": "Cyclone Gamane - north-east Madagascar floods",
        "start": "2024-03-26T00:00:00Z",
        "end": "2024-03-29T23:59:59Z",
    },
]

STEP_MINUTES = 15
WIDTH = 720
HEIGHT = 720
GIF_FPS = 6
MAX_FRAMES_PER_GIF = 96
OUTPUT_DIR = Path("meteosat_reflectance_wms")
PAUSE_SECONDS = 0.15

# Daytime visible reflectance is dark at night. Restrict to daylight hours over Madagascar if desired.
DAYLIGHT_ONLY = True
DAYLIGHT_UTC_START_HOUR = 3
DAYLIGHT_UTC_END_HOUR = 15

## 3. WMS layer discovery

In [ ]:
def local_name(tag):
    return tag.rsplit("}", 1)[-1]


def first_child_text(element, child_name):
    for child in element:
        if local_name(child.tag) == child_name and child.text:
            return child.text.strip()
    return None


def list_wms_layers(pattern="msg_iodc:(vis|rgb|hrv|ir108)|reflectance|natural"):
    response = requests.get(
        EUMETVIEW_WMS_URL,
        params={"SERVICE": "WMS", "VERSION": "1.3.0", "REQUEST": "GetCapabilities"},
        timeout=60,
    )
    response.raise_for_status()
    root = ET.fromstring(response.content)
    regex = re.compile(pattern, re.IGNORECASE) if pattern else None
    rows = []

    for layer in root.iter():
        if local_name(layer.tag) != "Layer":
            continue
        name = first_child_text(layer, "Name")
        if not name:
            continue
        title = first_child_text(layer, "Title") or ""
        text = f"{name} {title}"
        if regex is None or regex.search(text):
            rows.append({"name": name, "title": title})

    return pd.DataFrame(rows)


layers_df = list_wms_layers()
display(layers_df)

if REFLECTANCE_LAYER not in set(layers_df["name"]):
    print("Selected layer not shown by the discovery filter. It may still exist, or adjust REFLECTANCE_LAYER.")

## 4. Download reflectance frames

In [ ]:
def parse_datetime(value):
    value = value.replace("Z", "+00:00")
    dt = datetime.fromisoformat(value)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def safe_name(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")


def iter_times(start, end, step_minutes):
    current = start
    step = timedelta(minutes=step_minutes)
    while current <= end:
        if not DAYLIGHT_ONLY or DAYLIGHT_UTC_START_HOUR <= current.hour < DAYLIGHT_UTC_END_HOUR:
            yield current
        current += step


def split_times(times, max_frames):
    return [times[i:i + max_frames] for i in range(0, len(times), max_frames)]


def wms_params(layer, timestamp):
    return {
        "SERVICE": "WMS",
        "VERSION": "1.1.1",
        "REQUEST": "GetMap",
        "LAYERS": layer,
        "STYLES": "",
        "SRS": "EPSG:4326",
        "BBOX": ",".join(str(v) for v in BBOX),
        "WIDTH": str(WIDTH),
        "HEIGHT": str(HEIGHT),
        "FORMAT": "image/png",
        "TRANSPARENT": "false",
        "TIME": timestamp.strftime("%Y-%m-%dT%H:%M:%SZ"),
    }


def annotate_frame(path, label):
    image = PILImage.open(path).convert("RGBA")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    margin = 10
    padding = 6
    text_box = draw.textbbox((0, 0), label, font=font)
    box = (
        margin,
        margin,
        margin + text_box[2] - text_box[0] + 2 * padding,
        margin + text_box[3] - text_box[1] + 2 * padding,
    )
    draw.rectangle(box, fill=(0, 0, 0, 170))
    draw.text((margin + padding, margin + padding), label, fill=(255, 255, 255, 255), font=font)
    image.convert("RGB").save(path)


def download_frame(layer, timestamp, output_path):
    response = requests.get(EUMETVIEW_WMS_URL, params=wms_params(layer, timestamp), timeout=90)
    response.raise_for_status()
    content_type = response.headers.get("content-type", "")
    if "xml" in content_type.lower() or response.content[:100].lstrip().startswith(b"<"):
        raise RuntimeError(response.text[:1000])

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_bytes(response.content)
    annotate_frame(output_path, f"{layer} {timestamp:%Y-%m-%d %H:%M UTC}")
    return output_path

## 5. Build GIF animations

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
gif_paths = []
frame_rows = []

for event in FLOOD_EVENTS:
    event_name = safe_name(event["name"])
    start = parse_datetime(event["start"])
    end = parse_datetime(event["end"])
    times = list(iter_times(start, end, STEP_MINUTES))
    chunks = split_times(times, MAX_FRAMES_PER_GIF)
    print(event["name"], "frames:", len(times), "chunks:", len(chunks))

    for chunk_idx, chunk_times in enumerate(chunks, start=1):
        frame_paths = []
        frame_dir = OUTPUT_DIR / event_name / f"part_{chunk_idx:02d}"

        for timestamp in chunk_times:
            frame_path = frame_dir / f"{timestamp:%Y%m%dT%H%M%SZ}.png"
            try:
                download_frame(REFLECTANCE_LAYER, timestamp, frame_path)
                frame_paths.append(frame_path)
                frame_rows.append({"event": event["name"], "time": timestamp, "frame": str(frame_path)})
                print("downloaded", frame_path)
            except Exception as exc:
                print("failed", timestamp, exc)
            if PAUSE_SECONDS:
                time.sleep(PAUSE_SECONDS)

        if frame_paths:
            images = [imageio.imread(path) for path in frame_paths]
            suffix = f"_part{chunk_idx:02d}" if len(chunks) > 1 else ""
            gif_path = OUTPUT_DIR / f"{event_name}{suffix}_{REFLECTANCE_LAYER.replace(':', '_')}.gif"
            imageio.mimsave(gif_path, images, duration=1 / GIF_FPS)
            gif_paths.append(gif_path)
            print("GIF:", gif_path)
            display(Image(filename=str(gif_path)))

frames_df = pd.DataFrame(frame_rows)
display(pd.DataFrame({"gif": [str(p) for p in gif_paths]}))

## 6. Optional: WEkEO HDA metadata for raw SEVIRI image data

WEkEO HDA currently exposes many MSG/IODC derived products, but the raw High Rate SEVIRI IODC product may return `404` depending on the catalogue exposure/account. This optional cell checks the raw reflectance/radiance product id only. It does not use rainfall products.

In [ ]:
RUN_WEKEO_HDA_CHECK = False
RAW_SEVIRI_DATASET_ID = "EO:EUM:DAT:MSG:HRSEVIRI-IODC"

if RUN_WEKEO_HDA_CHECK:
    from hda import Client, Configuration

    username = os.environ.get("WEKEO_USERNAME") or input("WEkEO username: ")
    password = os.environ.get("WEKEO_PASSWORD") or getpass.getpass("WEkEO password: ")
    hda_client = Client(config=Configuration(user=username, password=password))

    try:
        dataset_info = hda_client.dataset(RAW_SEVIRI_DATASET_ID)
        print("Raw SEVIRI dataset available in WEkEO HDA:", RAW_SEVIRI_DATASET_ID)
        display(dataset_info)
    except Exception as exc:
        print("Raw SEVIRI dataset is not exposed through this WEkEO HDA endpoint/account:", RAW_SEVIRI_DATASET_ID)
        print(type(exc).__name__, exc)

## Notes

- This notebook uses reflectance/visible WMS imagery by default (`msg_iodc:vis006`), not precipitation.
- `msg_iodc:rgb_natural` and `msg_iodc:rgb_eview` are useful rendered visible/RGB alternatives.
- Visible reflectance is daytime-only; keep `DAYLIGHT_ONLY = True` for cleaner animations.
- For night-time cloud motion use `msg_iodc:ir108`, but that is thermal infrared brightness, not reflectance.
- The WEkEO HDA optional cell checks raw SEVIRI availability only and deliberately avoids rainfall datasets.